# Minimal RAG with Evaluation

## 1. Setup

In [ ]:
!pip install datasets sentence-transformers faiss-cpu tqdm

## 2. Load Dataset

In [1]:
from datasets import load_dataset

docs = load_dataset('rag-datasets/rag-mini-wikipedia', 
                    'text-corpus',
                    split="passages")

texts = [d["passage"] for d in docs]

print(f'Loaded {len(texts)} documents')

Loaded 3200 documents


In [8]:
import pandas as pd
pd.set_option('display.max_colwidth', None)


df = pd.DataFrame(docs)
df.head()

,passage,id
0,"Uruguay (official full name in ; pron. , Eastern Republic of Uruguay) is a country located in the southeastern part of South America. It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.",0
1,"It is bordered by Brazil to the north, by Argentina across the bank of both the Uruguay River to the west and the estuary of RÃ­o de la Plata to the southwest, and the South Atlantic Ocean to the southeast. It is the second smallest independent country in South America, larger only than Suriname and the French overseas department of French Guiana.",1
2,"Montevideo was founded by the Spanish in the early 18th century as a military stronghold. Uruguay won its independence in 1828 following a three-way struggle between Spain, Argentina and Brazil. It is a constitutional democracy, where the president fulfills the roles of both head of state and head of government",2
3,"The economy is largely based in agriculture (making up 10% of the GDP and the most substantial export) and the state-sector, and relies heavily on world trade. Consequently, it is badly affected by any downturn in global prices. However, the economy is on the whole more stable than surrounding states, and it maintains a solid reputation with investors.",3
4,"According to Transparency International, Uruguay is the second least corrupt country in Latin America (after Chile), Transparency.org. with its political and labor conditions being among the freest on the continent.",4


In [3]:
texts[0:5]

['Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.',
 'It is bordered by Brazil to the north, by Argentina across the bank of both the Uruguay River to the west and the estuary of RÃ\xado de la Plata to the southwest, and the South Atlantic Ocean to the southeast. It is the second smallest independent country in South America, larger only than Suriname and the French overseas department of French Guiana.',
 'Montevideo was founded by the Spanish in the early 18th century as a military stronghold. Uruguay won its independence in 1828 following a three-way struggle between Spain, Argentina and Brazil. It is a constitutional democracy, where the president fulfills the roles of both head of state and head of government',
 'The economy is largely based in agriculture (making up 10% of the GDP

In [4]:
# QA (for evaluation)
qa = load_dataset(
    "rag-datasets/rag-mini-wikipedia",
    "question-answer",
    split="test"
)

In [5]:
print(qa[0:5])
print(qa.column_names)
len(qa)

{'question': ['Was Abraham Lincoln the sixteenth President of the United States?', 'Did Lincoln sign the National Banking Act of 1863?', 'Did his mother die of pneumonia?', "How many long was Lincoln's formal education?", 'When did Lincoln begin his political career?'], 'answer': ['yes', 'yes', 'no', '18 months', '1832'], 'id': [0, 2, 4, 6, 8]}
['question', 'answer', 'id']


918

## 3. Create Embeddings

In [6]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embed_model.encode(texts, 
                                    batch_size=32,
                                    show_progress_bar=True)

/home/amir/miniconda3/envs/llm/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
doc_embeddings.shape

(3200, 384)

## 4. Build Vector Index (FAISS)

In [8]:
import faiss
import numpy as np

dim = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)

index.add(np.array(doc_embeddings))

print('FAISS index built')

FAISS index built


In [9]:
# faiss.IndexFlatIP

## 5. Retrieval Function

In [10]:
def retrieve(query, k=3):
    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k)
    return [texts[i] for i in I[0]] #, I[0], D[0]

## 6. Prompt Builder

In [11]:
def build_prompt(query, contexts):
    context_block = '\n\n'.join([c for c in contexts])
    return f"""
            Answer the question using ONLY the context.

            Context:
            {context_block}

            Question:
            {query}
            Answer:
            """

## 7. Demo Query

In [12]:
query = 'Was the sixteenth President of the United States Abraham Lincoln?'
query = "Is the ball red?"
contexts = retrieve(query, k=5)

for i, text in enumerate(contexts):
    print(f'\n--- Context {i+1} ---\n{text[:300]}...')

prompt = build_prompt(query, contexts)

print('\n--- Prompt sent to LLM ---\n')
print(prompt)


--- Context 1 ---
Red Kangaroo (Macropus rufus)...

--- Context 2 ---
An infrared image of a polarbear....

--- Context 3 ---
He also showed that the coloured light does not change its properties, by separating out a coloured beam and shining it on various objects.  Newton noted that regardless of whether it was reflected or scattered or transmitted, it stayed the same colour. Thus the colours we observe are the result of ...

--- Context 4 ---
* The Eastern Grey Kangaroo (Macropus giganteus) is less well-known than the red (outside of Australia), but the most often seen, as its range covers the fertile eastern part of the continent. ...

--- Context 5 ---
Egyptian Flag Until 1958....

--- Prompt sent to LLM ---


            Answer the question using ONLY the context.

            Context:
            Red Kangaroo (Macropus rufus)

An infrared image of a polarbear.

He also showed that the coloured light does not change its properties, by separating out a coloured beam and shining it

In [13]:
context_block = '\n\n'.join([c for c in contexts])
context_block

"Red Kangaroo (Macropus rufus)\n\nAn infrared image of a polarbear.\n\nHe also showed that the coloured light does not change its properties, by separating out a coloured beam and shining it on various objects.  Newton noted that regardless of whether it was reflected or scattered or transmitted, it stayed the same colour. Thus the colours we observe are the result of how objects interact with the incident already-coloured light, not the result of objects generating the colour. For more details, see Newton's theory of colour.\n\n* The Eastern Grey Kangaroo (Macropus giganteus) is less well-known than the red (outside of Australia), but the most often seen, as its range covers the fertile eastern part of the continent. \n\nEgyptian Flag Until 1958."

In [14]:
for sample in qa.select(range(5)):
    print(sample)

{'question': 'Was Abraham Lincoln the sixteenth President of the United States?', 'answer': 'yes', 'id': 0}
{'question': 'Did Lincoln sign the National Banking Act of 1863?', 'answer': 'yes', 'id': 2}
{'question': 'Did his mother die of pneumonia?', 'answer': 'no', 'id': 4}
{'question': "How many long was Lincoln's formal education?", 'answer': '18 months', 'id': 6}
{'question': 'When did Lincoln begin his political career?', 'answer': '1832', 'id': 8}


In [15]:
for sample in qa.select(range(5)):
    q = sample['question']
    a = sample['answer']
    context = retrieve(q, 3)

    print('\n===========================')
    print('Q:', q)
    print('A:', a)

    print('\nRetrieved contexts:')
    for i, c in enumerate(context):
        print(f'\n--- Context {i+1} ---\n{c[:300]}...')

    # hit = any(a.lower() in c.lower() for c in ctx)
    # print('Retrieval hit:', hit)


Q: Was Abraham Lincoln the sixteenth President of the United States?
A: yes

Retrieved contexts:

--- Context 1 ---
Young Abraham Lincoln...

--- Context 2 ---
Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from March 4, 1861 until his assassination. As an outspoken opponent of the expansion of slavery in the United States, "[I]n his short autobiography written for the 1860 presidential camp...

--- Context 3 ---
Sixteen months before his death, his son, John Quincy Adams, became the sixth President of the United States (1825 1829), the only son of a former President to hold the office until George W. Bush in 2001....

Q: Did Lincoln sign the National Banking Act of 1863?
A: yes

Retrieved contexts:

--- Context 1 ---
Lincoln believed in the Whig theory of the presidency, which left Congress to write the laws while he signed them, vetoing only those bills that threatened his war powers. Thus, he signed the Homestead Act

## 8. Evaluation Dataset

In [16]:
eval_queries = [
    ('Who was Alan Turing?', 'mathematician'),
    ('What is the capital of France?', 'Paris'),
    ('Who developed the theory of relativity?', 'Einstein'),
    ('What is photosynthesis?', 'process'),
]

## 9. Answerer #1 — OpenAI API

Send the retrieved context + question to a hosted model (`gpt-4o-mini`) and read back the generated answer.

In [17]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('/home/amir/source/.env')
oa_client = OpenAI()

OPENAI_MODEL = 'gpt-4o-mini'

def answer_with_openai(query, contexts, model=OPENAI_MODEL):
    prompt = build_prompt(query, contexts)
    resp = oa_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

# Sanity check on one sample
sample = qa[0]
ctx = retrieve(sample['question'], k=3)
print('Q:        ', sample['question'])
print('Gold A:   ', sample['answer'])
print('OpenAI A: ', answer_with_openai(sample['question'], ctx))

Q:         Was Abraham Lincoln the sixteenth President of the United States?
Gold A:    yes
OpenAI A:  Yes, Abraham Lincoln was the sixteenth President of the United States.


## 10. Answerer #2 — Local Model

Run an instruction-tuned model on the local machine via HuggingFace `transformers`. We use `google/flan-t5-base` — small enough to run on CPU, instruction-tuned for question answering.

In [18]:
from transformers import pipeline

LOCAL_MODEL = 'google/flan-t5-base'
local_pipe = pipeline('text2text-generation', model=LOCAL_MODEL)

def answer_with_local(query, contexts):
    prompt = build_prompt(query, contexts)
    out = local_pipe(prompt, max_new_tokens=64, do_sample=False)
    return out[0]['generated_text'].strip()

# Sanity check on the same sample
print('Q:        ', sample['question'])
print('Gold A:   ', sample['answer'])
print('Local A:  ', answer_with_local(sample['question'], ctx))

/home/amir/miniconda3/envs/llm/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Q:         Was Abraham Lincoln the sixteenth President of the United States?
Gold A:    yes
Local A:   yes


## 11. Compare OpenAI vs Local Model

For each evaluation question we:
1. Retrieve top-k contexts.
2. Generate an answer with each model.
3. Score both answers by cosine similarity between their embedding and the gold answer's embedding.

Higher mean similarity = better alignment with the reference answer.

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_match(pred, target):
    emb1 = embed_model.encode([pred])
    emb2 = embed_model.encode([target])
    return float(cosine_similarity(emb1, emb2)[0][0])

samples = qa.select(range(10))
rows = []
for s in samples:
    q, gold = s['question'], s['answer']
    contexts = retrieve(q, k=3)
    a_oa = answer_with_openai(q, contexts)
    a_loc = answer_with_local(q, contexts)
    rows.append({
        'question': q,
        'gold': gold,
        'openai': a_oa,
        'local': a_loc,
        'sim_openai': semantic_match(a_oa, gold),
        'sim_local': semantic_match(a_loc, gold),
    })

results_df = pd.DataFrame(rows)
print(results_df[['question', 'gold', 'openai', 'local', 'sim_openai', 'sim_local']].to_string(index=False))
print('\nMean semantic similarity to gold answer:')
print(f"  OpenAI ({OPENAI_MODEL}): {results_df['sim_openai'].mean():.3f}")
print(f"  Local  ({LOCAL_MODEL}): {results_df['sim_local'].mean():.3f}")

Token indices sequence length is longer than the specified maximum sequence length for this model (649 > 512). Running this sequence through the model will result in indexing errors


                                                         question                                                                      gold                                                                                                              openai              local  sim_openai  sim_local
Was Abraham Lincoln the sixteenth President of the United States?                                                                       yes                                              Yes, Abraham Lincoln was the sixteenth President of the United States.                yes    0.157542   1.000000
               Did Lincoln sign the National Banking Act of 1863?                                                                       yes                                                               Yes, Lincoln signed the National Banking Act of 1863.                Yes    0.100682   1.000000
                                 Did his mother die of pneumonia?                                         

In [20]:
results_df

,question,gold,openai,local,sim_openai,sim_local
0,Was Abraham Lincoln the sixteenth President of...,yes,"Yes, Abraham Lincoln was the sixteenth Preside...",yes,0.157542,1.000000
1,Did Lincoln sign the National Banking Act of 1...,yes,"Yes, Lincoln signed the National Banking Act o...",Yes,0.100682,1.000000
2,Did his mother die of pneumonia?,no,"No, his mother died of Typhoid fever.",No,0.200403,1.000000
3,How many long was Lincoln's formal education?,18 months,Lincoln's formal education consisted of about ...,18 months,0.364170,1.000000
4,When did Lincoln begin his political career?,1832,Lincoln began his political career in 1832.,1832,0.553563,1.000000
5,What did The Legal Tender Act of 1862 establish?,"the United States Note, the first paper curren...",The Legal Tender Act of 1862 established the U...,United States Note,0.855398,0.629352
6,Who suggested Lincoln grow a beard?,11-year-old Grace Bedell,11-year-old Grace Bedell,Grace Bedell,1.000000,0.805405
7,When did the Gettysburg address argue that Ame...,1776,The Gettysburg Address argued that America was...,1776,0.573339,1.000000
8,Did Lincoln beat John C. Breckinridge in the 1...,yes,"Yes, Lincoln beat John C. Breckinridge in the ...",yes,0.112788,1.000000
9,Was Abraham Lincoln the first President of the...,No,"No, Abraham Lincoln was not the first Presiden...",was the sixteenth,0.141024,0.193899
